<a href="https://colab.research.google.com/github/bsheese/387/blob/main/21_training/21_1_Practical_Training/21_1_1_Overfitting_and_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training in Practice: Part 2
## Overfitting and Regularization

---

## What This Notebook Is About

In 19_1_1 you saw a model overfit: it kept getting better on its training photos while it stopped improving on new ones. That model only trained a last layer of 2,052 numbers, so the overfitting was mild. Networks with many more weights than training examples can overfit badly, and some of the most common tools in deep learning exist to stop them. They're called **regularization**.

This notebook makes a network overfit on purpose, then tries three fixes and compares them:

1. **Dropout**: switching off a random part of the network during training
2. **Weight decay**: pulling every weight a little toward zero at every step
3. **More data**: the same network, trained on ten times as many images

Along the way it explains `model.train()` and `model.eval()`, which 19_0_1 and 21_0_1 used without explanation.

**Optional reference.** If you took CS 377, you have seen these ideas before in [17_2_1_3 (Ridge and Lasso)](https://github.com/bsheese/cs377/blob/main/17_regression_crossval/17_2_1_MLR/17_2_1_3_MLR_Ames_Part3_Regularization.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

In [ ]:
import copy
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

data = np.load(fetch('fashion10k.npz'))
CLASSES = list(data['classes'])

def prepare(images, labels):
    # 28 x 28 pixels from 0-255  ->  784 numbers from 0 to 1, one row per image
    X = torch.tensor(images.reshape(len(images), -1) / 255, dtype=torch.float32)
    return X.to(device), torch.tensor(labels, dtype=torch.long).to(device)

X_train, y_train = prepare(data['x_train'], data['y_train'])
X_val, y_val = prepare(data['x_val'], data['y_val'])
X_test, y_test = prepare(data['x_test'], data['y_test'])
loss_fn = nn.CrossEntropyLoss()
print('device:', device, '  training images:', len(X_train))

# The training loop from 21_0_1
def train_one_epoch(model, loader, optimizer):
    model.train()
    for x, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        scores = model(X)
        return loss_fn(scores, y).item(), (scores.argmax(dim=1) == y).float().mean().item()

def make_loader(X, y, batch_size=64):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True)

def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in histories.items():
        epochs = range(1, len(h['val_loss']) + 1)
        line, = ax1.plot(epochs, h['val_loss'], label=f'{label}: validation')
        ax1.plot(epochs, h['train_loss'], '--', color=line.get_color(), label=f'{label}: training')
        ax2.plot(epochs, h['val_acc'], label=f'{label}: validation', color=line.get_color())
        ax2.plot(epochs, h['train_acc'], '--', color=line.get_color(), label=f'{label}: training')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(fontsize=8)
    ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def fit(model, optimizer, epochs, X, y):
    loader = make_loader(X, y)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        train_one_epoch(model, loader, optimizer)
        for name, Xs, ys in [('train', X, y), ('val', X_val, y_val)]:
            loss, acc = evaluate(model, Xs, ys)
            history[f'{name}_loss'].append(loss)
            history[f'{name}_acc'].append(acc)
    return history

The helper functions are the ones from 21_0_1, with one change: `fit` now takes the training images as arguments, because this notebook trains on different amounts of data.

---

## Section 1: Making a Network Overfit

To make overfitting easy to see, use a small training set and a big network. The small training set is the first 100 images of each class, 1,000 in all:

In [ ]:
small = torch.cat([torch.where(y_train == k)[0][:100] for k in range(10)])
X_small, y_small = X_train[small], y_train[small]
print('small training set:', X_small.shape)

def big_network(dropout=0.0):
    return nn.Sequential(
        nn.Linear(784, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 10),
    ).to(device)

print(f'{sum(p.numel() for p in big_network().parameters()):,} weights')

The network has two hidden layers of 512 neurons and about 670,000 weights: about 670 weights for every training image. (The `nn.Dropout` layers are explained in Section 2. With `dropout=0.0` they do nothing.) Train it for 60 epochs:

In [ ]:
torch.manual_seed(0)
model = big_network()
plain = fit(model, torch.optim.Adam(model.parameters(), lr=0.001), epochs=60, X=X_small, y=y_small)

best = int(np.argmin(plain['val_loss']))
print(f"training accuracy after 60 epochs:   {plain['train_acc'][-1]:.3f}")
print(f"validation accuracy after 60 epochs: {plain['val_acc'][-1]:.3f}")
print(f"validation loss: lowest {plain['val_loss'][best]:.3f} at epoch {best + 1}, {plain['val_loss'][-1]:.3f} at the end")
plot_history({'no regularization': plain})

This is overfitting, much more clearly than in 19_1_1:

- The training accuracy reaches nearly 1.0 and the training loss goes to almost 0. The network can classify its 1,000 training images almost perfectly.
- The validation accuracy stops improving at around 0.8 within the first ten or so epochs.
- The validation loss reaches its lowest point within the first ten or so epochs and then climbs steadily for the rest of training, ending far above its best value.

Why does the validation loss keep rising when the validation accuracy stays about the same? The loss measures confidence as well as correctness (19_1_4): a wrong answer given with probability 0.99 costs far more than one given with 0.6. As the network memorizes its training images it becomes more and more confident about everything, including the validation images it gets wrong. So a rising validation loss is often the first sign of overfitting, even before the accuracy drops.

With 670 weights per image, the network has more than enough capacity to fit details of these particular 1,000 images, a crease here, a background speck there, that don't carry over to other images.

---

## Section 2: Dropout

**Dropout** switches off a random fraction of the hidden values during each training step: `nn.Dropout(0.5)` sets each value that passes through it to 0 with probability 0.5, choosing again for every image and every batch. To keep the total about the same, the values that survive are doubled (in general, divided by $1 - p$).

In [ ]:
drop = nn.Dropout(0.5)
x = torch.ones(1, 10)

drop.train()
print('training mode:  ', drop(x))
print('training mode:  ', drop(x))
drop.eval()
print('evaluation mode:', drop(x))

In training mode a different half of the values is zeroed each time and the rest are doubled. In evaluation mode dropout does nothing at all, so every hidden neuron contributes.

Why would breaking the network on purpose help? Because no neuron can count on any particular other neuron being there, the network can't build fragile combinations that only work for specific training images. It has to spread what it learns across many neurons, and features that many neurons agree on tend to be the ones that carry over to new images. You can also think of each training step as training a different, randomly thinned network, with the full network at evaluation time acting like an average of all of them.

Here's the same network with dropout after each hidden layer, trained the same way:

In [ ]:
torch.manual_seed(0)
model = big_network(dropout=0.5)
dropout = fit(model, torch.optim.Adam(model.parameters(), lr=0.001), epochs=60, X=X_small, y=y_small)

for label, h in {'no regularization': plain, 'dropout 0.5': dropout}.items():
    print(f"{label:18} validation accuracy {h['val_acc'][-1]:.3f}   validation loss {h['val_loss'][-1]:.3f}")
plot_history({'no regularization': plain, 'dropout 0.5': dropout})

Dropout makes the clearest difference to the validation loss: it rises much more slowly and ends well below the unregularized network's. The validation accuracy improves only a little. The network still overfits (its training accuracy is still close to 1.0); dropout slows the overfitting down rather than preventing it. With only 1,000 training images, there's a limit to how well any network can do.

---

## Section 3: Why `model.train()` and `model.eval()` Matter

Dropout is the first layer you've met that behaves differently in training and evaluation, and `model.train()` and `model.eval()` switch every such layer in a model at once. Forgetting `model.eval()` before measuring a model would leave dropout switched on, so half the network would be missing, and the results would change every time you measured:

In [ ]:
def accuracy_in_mode(model, mode):
    model.train() if mode == 'train' else model.eval()
    with torch.no_grad():
        return (model(X_val).argmax(dim=1) == y_val).float().mean().item()

print('measured in training mode:  ', [round(accuracy_in_mode(model, 'train'), 3) for _ in range(3)])
print('measured in evaluation mode:', [round(accuracy_in_mode(model, 'eval'), 3) for _ in range(3)])

In training mode the same model on the same images gives a lower accuracy that changes from one measurement to the next. In evaluation mode the answer is the same every time. That's why `evaluate` calls `model.eval()`, and why `train_one_epoch` calls `model.train()` to switch dropout back on.

The other common layer with two behaviors is **batch normalization**, which 22_0 introduces because convolutional networks use it heavily. In training mode, a batch-norm layer keeps a running average of the values passing through it and updates that average with every batch; in evaluation mode it uses the stored average without changing it. This is why 19_0_1 kept ResNet-18 in evaluation mode the whole time, even while training. ResNet-18 contains 20 batch-norm layers (the `bn` weights in 20_1_1's list). In training mode they would have kept updating their stored averages to fit the Pets photos, even though their weights were frozen, and then the saved last layer alone wouldn't have been enough to rebuild the model.

---

## Section 4: Weight Decay

**Weight decay** shrinks every weight a little toward 0 at every step, in addition to the usual step downhill. A weight only stays large if the slopes keep pushing it there, so the network is pushed toward using many small weights instead of a few large ones that fit particular training images. In PyTorch it's an argument to the optimizer; `torch.optim.AdamW` is Adam with weight decay done the recommended way.

For 377 graduates: this is the same idea as Ridge regression, which adds a penalty for large coefficients to the loss.

In [ ]:
torch.manual_seed(0)
model = big_network()
decay = fit(model, torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1.0), epochs=60, X=X_small, y=y_small)

for label, h in {'no regularization': plain, 'weight decay 1.0': decay}.items():
    print(f"{label:18} validation accuracy {h['val_acc'][-1]:.3f}   validation loss {h['val_loss'][-1]:.3f}")
plot_history({'no regularization': plain, 'weight decay 1.0': decay})

Weight decay also holds the validation loss down, by less than dropout did here, and makes little difference to the accuracy. (The strength, 1.0 here, is a setting to tune on validation data like the learning rate; values from 0.01 to 1 are common with AdamW.) Dropout and weight decay are often used together, and 21_1_2 does that.

---

## Section 5: More Data

The fourth option isn't a technique at all: give the network more training images. Here's the same big network with no regularization, trained on all 10,000 training images instead of 1,000. Each epoch is ten times as long, so it gets 30 epochs.

In [ ]:
torch.manual_seed(0)
model = big_network()
more_data = fit(model, torch.optim.Adam(model.parameters(), lr=0.001), epochs=30, X=X_train, y=y_train)

results = {'no regularization (1,000 images)': plain, 'dropout 0.5 (1,000 images)': dropout,
           'weight decay 1.0 (1,000 images)': decay, 'no regularization (10,000 images)': more_data}
print(f"{'':36} {'best val acc':>13} {'final val acc':>14} {'final val loss':>15}")
for label, h in results.items():
    print(f"{label:36} {max(h['val_acc']):13.3f} {h['val_acc'][-1]:14.3f} {h['val_loss'][-1]:15.3f}")
plot_history({'1,000 images': plain, '10,000 images': more_data})

Ten times as much data raises the validation accuracy by several points, far more than dropout or weight decay did. That's the general rule: more data is the most reliable cure for overfitting, and regularization is what you use when you can't get more. (The 10,000-image network still overfits eventually; its validation loss starts rising after a while, because it's still a very big network for 10,000 images.)

There's also a fifth tool hiding in every plot in this notebook. In every run the validation loss was lowest partway through training, and the network was worse by the end. Stopping training at the epoch with the lowest validation loss, as 19_1_1 suggested, is called **early stopping**, and it's the simplest regularizer of all. 21_1_2 adds it to the training loop.

---

## Summary

- A network with far more weights than training examples can **overfit**: training accuracy near 1.0, validation accuracy stuck, and validation loss climbing steadily. Rising validation loss usually shows up before falling validation accuracy, because the network becomes confident about its mistakes.
- **Dropout** (`nn.Dropout(p)`) zeroes each hidden value with probability $p$ during training and scales up the rest. It forces the network to spread what it learns across many neurons. Here it held the validation loss well down and helped the accuracy a little.
- `model.train()` and `model.eval()` switch layers like dropout and batch normalization between their two behaviors. Measuring a model without `model.eval()` gives noisy, wrong results. Batch norm's running averages are why 19_0_1 kept ResNet-18 in evaluation mode.
- **Weight decay** (`torch.optim.AdamW(..., weight_decay=...)`) shrinks every weight toward 0 at every step; it's the network version of Ridge regression. Here it helped less than dropout.
- **More data** helped far more than either: ten times the images raised validation accuracy by several points.
- **Early stopping**, keeping the weights from the epoch with the lowest validation loss, is the subject of 21_1_2.

**Next:** 21_1_2 puts the tools together: early stopping with a saved checkpoint, a learning-rate schedule, and dropout and weight decay, to train the best network this unit can make on Fashion-MNIST.